# 🚗 Pipeline Lengkap Deteksi & Pengenalan Pelat Nomor Kendaraan (Tahap 1 - 8)
### Proyek Pengolahan Citra Digital dan Visi Komputer

Notebook ini mengeksekusi **seluruh tahapan sistem secara berurutan (Tahap 1 hingga Tahap 8)** menggunakan GPU A100 di Google Colab:
* **Tahap 1:** Setup Lingkungan & Mount Google Drive
* **Tahap 2:** Ekstraksi Dataset & Training YOLOv8 (**Materi P21: YOLO**)
* **Tahap 3:** Pengujian Preprocessing (**Materi P4: Grayscale & P6: CLAHE**)
* **Tahap 4:** Pengujian Segmentasi Biner (**Materi P12: Adaptive/Otsu Thresholding**)
* **Tahap 5:** Pengenalan Karakter OCR & Validasi Format Plat
* **Tahap 6:** Visualisasi Bounding Box, Garis Penunjuk & Badge Teks
* **Tahap 7:** Menjalankan Web App Streamlit Interaktif
* **Tahap 8:** Evaluasi Akurasi Pipeline terhadap Ground Truth (`labelswithLP`)

--- 
## 📌 TAHAP 1: Setup Lingkungan & Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Masuk ke direktori proyek di Google Drive
project_path = '/content/drive/MyDrive/Indonesian License Plate Detection'
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f'✅ Direktori aktif: {os.getcwd()}')
else:
    print(f'⚠️ Folder {project_path} tidak ditemukan!')

# Cek GPU & Install pustaka
!nvidia-smi
!pip install -q streamlit ultralytics easyocr pillow matplotlib

--- 
## 📌 TAHAP 2: Ekstraksi Dataset & Training YOLOv8 (Materi P21: YOLO)

In [ ]:
import os
import zipfile
import yaml
import glob

# 1. Ekstrak Dataset ZIP ke SSD Lokal Colab
zip_candidates = [
    '/content/drive/MyDrive/Indonesian License Plate Detection/Dataset/dataset_plate.zip',
    '/content/drive/MyDrive/Indonesian License Plate Detection/dataset_plate.zip'
]
zip_path = None
for c in zip_candidates:
    if os.path.exists(c):
        zip_path = c
        break

if not zip_path:
    raise FileNotFoundError("❌ dataset_plate.zip tidak ditemukan di Google Drive!")

print(f"📦 Mengekstrak {zip_path} ke SSD lokal /content/dataset/ ...")
os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')
print("✅ Ekstraksi dataset selesai!")

# 2. Deteksi Letak Folder 'images/train' Secara Otomatis
train_dirs = glob.glob('/content/dataset/**/images/train', recursive=True)
if not train_dirs:
    train_dirs = glob.glob('/content/dataset/**/train', recursive=True)
if not train_dirs:
    raise FileNotFoundError("❌ Folder train tidak ditemukan di dalam hasil ekstrak zip!")

extracted_root = train_dirs[0].split('/images')[0] if 'images' in train_dirs[0] else os.path.dirname(train_dirs[0])
print(f"📍 Root dataset terdeteksi: {extracted_root}")

# 3. Tulis Konfigurasi colab_data.yaml
colab_data_cfg = {
    'path': extracted_root,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'license_plate'}
}
with open('/content/colab_data.yaml', 'w') as f:
    yaml.dump(colab_data_cfg, f)
print("✅ Konfigurasi /content/colab_data.yaml siap!")

In [ ]:
from ultralytics import YOLO
import shutil

# 4. Training YOLOv8n di GPU A100 (~2-3 menit)
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/colab_data.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    device=0,
    name='plate_detector_colab',
    save=True
)

# 5. Simpan Bobot Terbaik ke Folder models/best.pt di Google Drive
best_weight = os.path.join(str(results.save_dir), 'weights', 'best.pt')
if os.path.exists(best_weight):
    os.makedirs('models', exist_ok=True)
    target_path = 'models/best.pt'
    shutil.copy(best_weight, target_path)
    print(f'🎉 Bobot terbaik berhasil disimpan ke: {target_path}')

--- 
## 📌 TAHAP 3: Uji Coba Preprocessing Citra Plat (Materi P4: Ruang Warna & P6: CLAHE)
Kita memotong area plat dari gambar uji coba, lalu menerapkan konversi ke Grayscale (P4) dan CLAHE (P6) untuk mempertajam kontras karakter.

In [ ]:
import importlib
import Script.detector, Script.preprocessor, Script.binarizer, Script.recognizer, Script.utils
importlib.reload(Script.detector)
importlib.reload(Script.preprocessor)
importlib.reload(Script.binarizer)
importlib.reload(Script.recognizer)
importlib.reload(Script.utils)

import cv2
import matplotlib.pyplot as plt
import glob
import random
from Script.detector import PlateDetector
from Script.preprocessor import PlatePreprocessor

# Cari gambar uji dari dataset test (mobil utuh)
test_images = [f for f in glob.glob('/content/dataset/**/images/test/*.jpg', recursive=True) if 'Recognition' not in f]
sample_img_path = random.choice(test_images) if test_images else 'sample.jpg'
print(f"🚗 Menguji gambar: {sample_img_path}")

# Deteksi Semua Plat pada Gambar (Materi P21 YOLO)
detector = PlateDetector('models/best.pt', conf_threshold=0.25)
preprocessor = PlatePreprocessor()
detections = detector.detect(sample_img_path)

if not detections:
    print("❌ Plat tidak terdeteksi pada gambar ini.")
else:
    print(f"🎉 Ditemukan {len(detections)} plat nomor pada gambar ini!\n")
    num_plates = len(detections)
    fig, axes = plt.subplots(num_plates, 4, figsize=(16, 3.8 * num_plates))
    if num_plates == 1:
        axes = [axes]
        
    for i, det in enumerate(detections):
        crop_img = det['crop']
        prep_res = preprocessor.process(crop_img)
        det['prep_res'] = prep_res  # Simpan di masing-masing objek deteksi agar tidak tertukar!
        
        print(f"Plat #{i+1} -> {prep_res['clahe_status']}")
        
        axes[i][0].imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
        axes[i][0].set_title(f"Plat #{i+1}: Crop Asli (P21)")
        axes[i][0].axis('off')
        
        axes[i][1].imshow(prep_res['grayscale'], cmap='gray')
        axes[i][1].set_title(f"Plat #{i+1}: Grayscale (P4)")
        axes[i][1].axis('off')
        
        axes[i][2].hist(prep_res['grayscale'].ravel(), bins=256, range=[0, 256], color='steelblue')
        axes[i][2].set_title(f"Histogram (std={prep_res['hist_std']})")
        
        axes[i][3].imshow(prep_res['enhanced'], cmap='gray')
        status_lbl = "CLAHE Aktif" if prep_res['clahe_applied'] else "Bypass CLAHE"
        axes[i][3].set_title(f"Output P6: {status_lbl}")
        axes[i][3].axis('off')
        
    plt.tight_layout()
    plt.show()


--- 
## 📌 TAHAP 4: Uji Coba Segmentasi Karakter (Materi P12: Thresholding)
Memisahkan karakter huruf dan angka dari latar belakang plat menjadi citra biner hitam-putih.

In [ ]:
from Script.binarizer import PlateBinarizer

binarizer = PlateBinarizer(method='otsu')

if detections:
    num_plates = len(detections)
    fig, axes = plt.subplots(num_plates, 2, figsize=(10, 3.5 * num_plates))
    if num_plates == 1:
        axes = [axes]
        
    for i, det in enumerate(detections):
        prep_res = det['prep_res']
        bin_res = binarizer.process(prep_res['enhanced'])
        det['bin_res'] = bin_res  # Simpan hasil binarisasi untuk masing-masing plat
        
        axes[i][0].imshow(bin_res['upscaled_gray'], cmap='gray')
        axes[i][0].set_title(f"Plat #{i+1}: Input Upscaled")
        axes[i][0].axis('off')
        
        axes[i][1].imshow(bin_res['binary_normalized'], cmap='gray')
        axes[i][1].set_title(f"Plat #{i+1}: Binarisasi Otsu (Materi P12)")
        axes[i][1].axis('off')
        
    plt.tight_layout()
    plt.show()


--- 
## 📌 TAHAP 5: Pembacaan Teks Karakter & Validasi Format Plat (EasyOCR)
Mengekstrak karakter alfanumerik dan memeriksa kepatuhan pola plat nomor Indonesia.

In [ ]:
from Script.recognizer import PlateRecognizer

recognizer = PlateRecognizer(use_gpu=True)

if detections:
    print("📋 HASIL PEMBACAAN OCR UNTUK SETIAP PLAT:\n")
    for i, det in enumerate(detections):
        bin_res = det['bin_res']
        ocr_res = recognizer.recognize(
            bin_res['binary_normalized'],
            fallback_image=bin_res['upscaled_gray'],
            extra_images=bin_res['variants'][:1]
        )

        det['ocr_res'] = ocr_res
        det['text'] = ocr_res['clean_text']

        valid = ocr_res['is_valid_format']
        low_conf = ocr_res['confidence'] < 0.4
        det['is_occluded'] = ocr_res['is_occluded'] or (not det['is_complete']) or (not valid) or low_conf

        if valid and not low_conf and det['is_complete']:
            status_str = "✅ LENGKAP & VALID"
        elif low_conf and not ocr_res['is_occluded']:
            status_str = "❓ TIDAK YAKIN (confidence rendah / bacaan tidak valid)"
        else:
            status_str = "⚠️ TERHALANG / TIDAK UTUH"

        print(f"--- Plat #{i+1} ---")
        print(f"• Teks Terbaca    : {det['text']}")
        print(f"• Confidence OCR  : {ocr_res['confidence']}")
        print(f"• Sumber Pemenang : {ocr_res['source']}")
        for name, fmt, conf in ocr_res['all_candidates']:
            print(f"    - {name:<32} -> {fmt:<14} (conf {conf})")
        print(f"• Rasio Aspek Box : {det['aspect_ratio']}")
        print(f"• Status          : {status_str}\n")


--- 
## 📌 TAHAP 6: Visualisasi Anotasi Bounding Box, Garis Penunjuk & Badge Teks

In [ ]:
from Script.utils import draw_plate_annotations

if detections:
    sample_bgr = cv2.imread(sample_img_path)
    annotated = draw_plate_annotations(sample_bgr, detections)
    
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    titles = [f"Plat #{i+1}: {d['text']}" for i, d in enumerate(detections)]
    plt.title(" | ".join(titles))
    plt.axis('off')
    plt.show()


--- 
## 📌 TAHAP 7: Menjalankan Aplikasi Web Streamlit di Colab

In [ ]:
print("👉 Password LocalTunnel Anda (copy nomor IP ini jika diminta saat membuka URL web):")
!curl -s ipv4.icanhazip.com

# Jalankan Streamlit di background dan buat tunnel publik
!streamlit run app.py & npx localtunnel --port 8501

--- 
## 📌 TAHAP 8: Evaluasi Akurasi Pipeline terhadap Ground Truth (labelswithLP)

In [ ]:
from Script.utils import get_ground_truth

test_files = [f for f in glob.glob('/content/dataset/**/images/test/*.jpg', recursive=True) if 'Recognition' not in f][:10]
print(f"Menguji {len(test_files)} sampel data test terhadap Ground Truth:\n")

total = len(test_files)
matched = 0

for fpath in test_files:
    fname = os.path.basename(fpath)
    gt = get_ground_truth(fname, os.getcwd())
    
    dets = detector.detect(fpath)
    if dets:
        c = preprocessor.process(dets[0]['crop'])
        b = binarizer.process(c['enhanced'])
        r = recognizer.recognize(b['binary_normalized'])
        pred_text = r['clean_text'].replace(' ', '')
    else:
        pred_text = 'TIDAK_TERDETEKSI'
        
    gt_clean = gt.replace(' ', '') if gt else 'N/A'
    is_match = (pred_text == gt_clean)
    if is_match:
        matched += 1
        
    status_sym = "✅ COCOK" if is_match else "⚠️ BEDA"
    print(f"File: {fname:<15} | Prediksi: {pred_text:<12} | Ground Truth: {gt_clean:<12} | {status_sym}")

print(f"\n🎯 Akurasi Pencocokan Sempurna (Exact Match): {matched}/{total} ({(matched/total)*100:.1f}%)")